# M6.2 · Özel autograd.Function

Kendi türevini sen yazdığın işlemler. Straight-through estimator, gradient reversal ve log-sum-exp'in gradyanlarını elle hesaplıyor, saf Python'da minik bir Function protokolü ve sonlu farklarla gradcheck yazıyor, NumPy'da softmax + cross-entropy'yi yalnız olasılıkları saklayan tek bir birleşik işlem olarak kuruyor, PyTorch'ta `torch.autograd.Function` ile aynı işlemleri `gradcheck`'ten geçiriyor ve en sonda yalnız girdisini saklayan, bellek dostu bir SiLU yazıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m6-2-custom-function.html](https://m-gonul.github.io/pytorch-lab/m6-2-custom-function.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m6_2_custom_function.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    LSE = need(ns, \"LogSumExp\", \"sınıf\")\n\n    def fw():\n        close(LSE.apply([1.0, 2.0])[0], math.log(math.e + math.e ** 2))\n        close(LSE.apply([1000.0, 1000.0])[0], 1000 + math.log(2), what=\"büyük girdi (taşma yok)\")\n    R.test(\"LogSumExp ileri (kararlı)\", fw)\n\n    def bw():\n        out, ctx = LSE.apply([1.0, 2.0, 3.0])\n        g = LSE.backward(ctx, 2.0)\n        e = [math.exp(x) for x in (1, 2, 3)]\n        close(g, [2 * v / sum(e) for v in e])\n        out, ctx = LSE.apply([1000.0, 999.0])\n        close(sum(LSE.backward(ctx, 1.0)), 1.0, what=\"büyük girdide softmax toplamı\")\n    R.test(\"LogSumExp geri = g · softmax\", bw)\n\n    def gc():\n        ok, err = need(ns, \"gradcheck\")(LSE, [0.3, -1.2, 2.0, 0.0])\n        ensure(ok, f\"LogSumExp gradcheck'ten geçmeli (hata {err:.2e})\")\n        ok2, err2 = ns[\"gradcheck\"](need(ns, \"RoundSTE\", \"sınıf\"), [0.3, 1.2, 2.7])\n        ensure(not ok2, \"RoundSTE gradcheck'ten GEÇMEMELİ: gradyanı bilerek gerçek türevden farklı\")\n        return f\"LSE hata {err:.1e}; STE hata {err2:.1f} (beklenen: gerçek türev 0, STE 1)\"\n    R.test(\"gradcheck: LSE geçer, STE geçmez\", gc)\n\n    def ste():\n        out, ctx = need(ns, \"RoundSTE\", \"sınıf\").apply([0.3, 1.7, -0.6])\n        ensure(out == [0.0, 2.0, -1.0], f\"ileri {out}\")\n        ensure(list(ns[\"RoundSTE\"].backward(ctx, [2.0, 3.0, 4.0])) == [2.0, 3.0, 4.0], \"geri özdeşlik\")\n    R.test(\"RoundSTE\", ste)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"ce_forward/ce_backward döngüsüz\", lambda: (max_loops(src, 0, func=\"ce_forward\"), max_loops(src, 0, func=\"ce_backward\")))\n    R.test(\"dir_gradcheck: en fazla tek döngü\", lambda: max_loops(src, 1, func=\"dir_gradcheck\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    Z, y = rng.normal(size=(8, 5)), rng.integers(0, 5, 8)\n\n    def fw():\n        loss, cache = need(ns, \"ce_forward\")(Z, y)\n        p = np.exp(Z) / np.exp(Z).sum(1, keepdims=True)\n        close(loss, -np.log(p[np.arange(8), y]).mean())\n        ensure(len(cache) == 2, \"cache yalnız (olasılıklar, y) olmalı\")\n        l2, _ = ns[\"ce_forward\"](Z + 1000, y)\n        close(l2, loss, what=\"logit'ler + 1000 (kararlılık)\")\n    R.test(\"ce_forward (kararlı)\", fw)\n\n    def bw():\n        _, cache = need(ns, \"ce_forward\")(Z, y)\n        d = need(ns, \"ce_backward\")(cache)\n        h = 1e-6; num = np.zeros_like(Z)\n        for i in range(8):\n            for j in range(5):\n                Zp = Z.copy(); Zp[i, j] += h; Zm = Z.copy(); Zm[i, j] -= h\n                num[i, j] = (ns[\"ce_forward\"](Zp, y)[0] - ns[\"ce_forward\"](Zm, y)[0]) / (2 * h)\n        close(d, num, atol=1e-7)\n    R.test(\"ce_backward = sayısal türev\", bw)\n\n    def dg():\n        f = lambda z: ns[\"ce_forward\"](z, y)[0]\n        good = lambda z: ns[\"ce_backward\"](ns[\"ce_forward\"](z, y)[1])\n        bad = lambda z: good(z) * 1.01\n        e1 = need(ns, \"dir_gradcheck\")(f, good, Z, np.random.default_rng(1))\n        e2 = ns[\"dir_gradcheck\"](f, bad, Z, np.random.default_rng(1))\n        ensure(e1 < 1e-5, f\"doğru gradyanda hata {e1:.2e}\")\n        ensure(e2 > 5e-3, f\"%1 hatalı gradyanı yakalamalı (hata {e2:.2e})\")\n        return f\"doğru gradyan {e1:.1e}, %1 bozuk gradyan {e2:.1e}\"\n    R.test(\"dir_gradcheck hatayı yakalıyor\", dg)\n",
 "d": "import torch\nfrom torch.autograd import gradcheck\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    LSE = need(ns, \"LogSumExp\", \"sınıf\")\n\n    def fw():\n        x = torch.randn(3, 4, 5, dtype=torch.float64)\n        close(LSE.apply(x), torch.logsumexp(x, -1))\n        big = torch.tensor([[1000.0, 1000.0, 0.0]])\n        ensure(torch.isfinite(LSE.apply(big)).all(), \"büyük girdide sonlu olmalı\")\n    R.test(\"LogSumExp ileri = torch.logsumexp\", fw)\n\n    def gc():\n        x = torch.randn(3, 6, dtype=torch.float64, requires_grad=True)\n        ensure(gradcheck(LSE.apply, (x,)), \"gradcheck\")\n        b = torch.tensor([[1000.0, 999.0]], dtype=torch.float64, requires_grad=True)\n        LSE.apply(b).sum().backward()\n        close(b.grad, torch.softmax(b.detach(), -1), what=\"büyük girdide gradyan\")\n    R.test(\"LogSumExp gradcheck + kararlılık\", gc)\n\n    def ste():\n        x = torch.tensor([0.3, 1.7, -0.6], requires_grad=True)\n        y = need(ns, \"RoundSTE\", \"sınıf\").apply(x)\n        close(y, [0.0, 2.0, -1.0])\n        (y * torch.tensor([2.0, 3.0, 4.0])).sum().backward()\n        close(x.grad, [2.0, 3.0, 4.0], what=\"STE gradyanı\")\n    R.test(\"RoundSTE\", ste)\n\n    def gr():\n        x = torch.tensor([1.0, -2.0], requires_grad=True)\n        y = need(ns, \"GradReverse\", \"sınıf\").apply(x, 0.5)\n        close(y, x.detach(), what=\"ileri özdeşlik\")\n        (y * torch.tensor([2.0, 3.0])).sum().backward()\n        close(x.grad, [-1.0, -1.5])\n    R.test(\"GradReverse\", gr)\n\n    def train_ste():\n        torch.manual_seed(0)\n        w = torch.randn(4, requires_grad=True)\n        target = torch.tensor([1.0, -2.0, 3.0, 0.0])\n        opt = torch.optim.SGD([w], lr=0.1)\n        for _ in range(200):\n            loss = ((ns[\"RoundSTE\"].apply(w) - target) ** 2).sum()\n            opt.zero_grad(); loss.backward(); opt.step()\n        close(torch.round(w.detach()), target, what=\"kuantize ağırlıklar hedefe ulaşmalı\")\n        return f\"quantization-aware: round(w) = {torch.round(w.detach()).tolist()}\"\n    R.test(\"STE ile kuantize parametre öğrenme\", train_ste)\n",
 "e": "import torch\nimport torch.nn.functional as F\nfrom torch.autograd import gradcheck\nfrom labcheck import close, need, ensure, bench, fmt_time\n\n\ndef saved_bytes(fn, x):\n    tot = [0]\n\n    def pack(t):\n        tot[0] += t.numel() * t.element_size()\n        return t\n    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):\n        fn(x)\n    return tot[0]\n\n\ndef check(ns, R):\n    f = need(ns, \"mem_eff_silu\")\n\n    def gc():\n        x = torch.randn(20, dtype=torch.float64, requires_grad=True)\n        ensure(gradcheck(f, (x,)), \"gradcheck\")\n        close(f(x), F.silu(x), what=\"ileri\")\n    R.test(\"gradcheck ve ileri = F.silu\", gc)\n\n    def mem():\n        x = torch.randn(1024, 1024, requires_grad=True)\n        naive = saved_bytes(lambda t: t * torch.sigmoid(t), x)\n        mine = saved_bytes(f, x)\n        ref = saved_bytes(F.silu, x)\n        ensure(mine == x.numel() * 4, f\"yalnız x saklanmalı: {mine:,} bayt\")\n        R.table(\"Saklanan bayt (1024 × 1024 fp32)\", [\"yöntem\", \"bayt\"],\n                [[\"x * sigmoid(x)\", f\"{naive:,}\"], [\"F.silu (yerleşik)\", f\"{ref:,}\"], [\"senin MemEffSiLU\", f\"{mine:,}\"]])\n    R.test(\"yalnız girdi saklanıyor\", mem)\n    x = torch.randn(2048, 2048, requires_grad=True)\n\n    def run(fn):\n        x.grad = None\n        fn(x).sum().backward()\n    t_naive = bench(lambda: run(lambda t: t * torch.sigmoid(t)), repeat=3)\n    t_mine = bench(lambda: run(f), repeat=3)\n    t_ref = bench(lambda: run(F.silu), repeat=3)\n    R.table(\"İleri + geri, 2048 × 2048\", [\"yöntem\", \"süre\"],\n            [[\"x * sigmoid(x)\", fmt_time(t_naive)], [\"senin MemEffSiLU\", fmt_time(t_mine)], [\"F.silu\", fmt_time(t_ref)]],\n            note=\"Yeniden hesaplama (sigmoid'i geri geçişte tekrar) az bellek karşılığında biraz ek iş demek; elementwise işlemlerde bu genelde ucuz.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m6_2_custom_function as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

`torch.autograd.Function` ile bir işlemin ileri ve geri geçişini **ayrı ayrı** tanımlarsın:

```python
class Exp(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        y = x.exp()
        ctx.save_for_backward(y)     # geri geçiş için yalnız bu
        return y

    @staticmethod
    def backward(ctx, grad_out):
        (y,) = ctx.saved_tensors
        return grad_out * y          # forward'un her girdisi için bir gradyan

y = Exp.apply(x)
```

Neden?

- **Sayısal kararlılık:** log-sum-exp'in türevi softmax; çıktıyı saklayıp $e^{x-\text{lse}}$ ile hesaplamak taşmaz.
- **Bilerek farklı gradyan:** straight-through estimator (ileri: round, geri: özdeşlik), gradient reversal (ileri: özdeşlik, geri: $-\lambda g$).
- **Bellek:** autograd'ın sakladıklarını azaltıp geri geçişte yeniden hesaplamak.
- **Birleşik (fused) işlemler:** softmax + cross-entropy'nin gradyanı $(p-y)/N$; ara grafı hiç kurmamak.

## 1 · Elle

STE'de geri geçiş "round yokmuş gibi" davranır: $\partial L/\partial x_i=w_i$. Gradient reversal ileri geçişte hiçbir şey yapmaz; geri geçişte işareti çevirip $\lambda$ ile ölçekler. Domain adaptation'da özellik çıkarıcının, alan ayırıcısını **şaşırtacak** yöne gitmesini sağlar.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$x=(0.3,\ 1.7)$, $w=(2,\ 3)$, $L=\sum_i\operatorname{round}(x_i)\,w_i$**

- `L`: $L$
- `true0`: Gerçek $\partial L/\partial x_1$
- `ste0`: STE ile $\partial L/\partial x_1$ (geri geçişte round = özdeşlik)
- `gr0`: Gradient reversal ($\lambda=0.5$), $L=\sum\operatorname{GR}(x_i)w_i$: $\partial L/\partial x_1$

**$\operatorname{lse}(x)=\log\sum_ie^{x_i}$, $x=(1,\ 2)$**

- `lse`: $\operatorname{lse}(x)$
- `g2`: $\partial\operatorname{lse}/\partial x_2=e^{x_2-\operatorname{lse}}$

In [ ]:
cevaplar = dict(
    L=None,
    true0=None,
    ste0=None,
    gr0=None,
    lse=None,
    g2=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"L":[6,0],"true0":[0,0],"ste0":[2,0],"gr0":[-1,0],"lse":[2.313262,0.0005],"g2":[0.731059,0.0005]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python. Hazır: `Ctx` (`save_for_backward`, `saved`) ve `Function.apply(*args) → (çıktı, ctx)`.

- `LogSumExp` — `forward(ctx, xs)` (liste → skaler, kararlı: max çıkar), `backward(ctx, g)` → liste.
- `RoundSTE` — `forward` eleman eleman `round`, `backward` gradyanı aynen geçirir.
- `gradcheck(fn_cls, xs, eps=1e-6, tol=1e-5)` → `(geçti_mi, en_büyük_hata)`. Skaler çıktılı fonksiyonlar için: `backward(ctx, 1.0)` ile analitik gradyan, merkezi farkla sayısal gradyan.

In [ ]:
%%writefile stage_b.py
import math


class Ctx:
    def __init__(self):
        self.saved = ()

    def save_for_backward(self, *xs):
        self.saved = xs


class Function:
    @classmethod
    def apply(cls, *args):
        ctx = Ctx()
        return cls.forward(ctx, *args), ctx


# ---- senin kodun ----

class LogSumExp(Function):
    @staticmethod
    def forward(ctx, xs):
        ...

    @staticmethod
    def backward(ctx, g):
        ...


class RoundSTE(Function):
    @staticmethod
    def forward(ctx, xs):
        ...

    @staticmethod
    def backward(ctx, g):
        ...


def gradcheck(fn_cls, xs, eps=1e-6, tol=1e-5):
    ...


print(LogSumExp.apply([1.0, 2.0])[0])        # 2.3133
print(gradcheck(LogSumExp, [1.0, 2.0, -0.5]))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, birleşik softmax + cross-entropy. **Döngü yok.**

- `ce_forward(logits, y)` → `(kayıp, cache)`; `logits: (N, C)`, `y: (N,)`. Kararlı log-softmax. `cache` yalnız olasılıkları ve `y`'yi tutsun.
- `ce_backward(cache)` → `dlogits = (p - onehot(y)) / N`.

Bir de `dir_gradcheck(f, grad, x, rng, k=10, eps=1e-6)` — rastgele $k$ yön $v$ için $\frac{f(x+\epsilon v)-f(x-\epsilon v)}{2\epsilon}$ ile $\langle\nabla f, v\rangle$'yi karşılaştırır, en büyük göreli hatayı döndürür. (Yönler üzerinde tek döngü serbest.) Tam gradcheck'in $O(d)$ maliyeti yerine $O(k)$.

In [ ]:
%%writefile stage_c.py
import numpy as np


def ce_forward(logits, y):
    ...


def ce_backward(cache):
    ...


def dir_gradcheck(f, grad, x, rng, k=10, eps=1e-6):
    ...


rng = np.random.default_rng(0)
Z, y = rng.normal(size=(5, 4)), rng.integers(0, 4, 5)
loss, cache = ce_forward(Z, y)
print(loss, ce_backward(cache).sum(1))     # satır toplamları 0

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

`torch.autograd.Function` alt sınıfları:

- `LogSumExp` — son eksende; `forward` çıktıyı (ve gerekirse girdiyi) saklar; `backward` = `g[..., None] * exp(x - out[..., None])`.
- `RoundSTE` — ileri `torch.round`, geri özdeşlik.
- `GradReverse` — `apply(x, lam)`; ileri `x.view_as(x)`, geri `-lam * g` ve `lam` için `None`.

Kontrol `torch.autograd.gradcheck` ile ve $x=1000$ gibi büyük girdilerle sınar.

In [ ]:
%%writefile stage_d.py
import torch


class LogSumExp(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ...

    @staticmethod
    def backward(ctx, g):
        ...


class RoundSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ...

    @staticmethod
    def backward(ctx, g):
        ...


class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ...

    @staticmethod
    def backward(ctx, g):
        ...


x = torch.tensor([[1.0, 2.0], [1000.0, 1000.0]], requires_grad=True)
y = LogSumExp.apply(x)
y.sum().backward()
print(y, x.grad)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`MemEffSiLU` — $\operatorname{silu}(x)=x\,\sigma(x)$. Türev: $\sigma(x)\big(1+x(1-\sigma(x))\big)$.

`forward`'da **yalnız `x`'i** sakla; $\sigma(x)$'i geri geçişte yeniden hesapla. Naif `x * torch.sigmoid(x)` üç tensör saklar (sigmoid çıktısı ve mul'un iki girdisi).

`mem_eff_silu(x)` = `MemEffSiLU.apply(x)`. Kontrol gradcheck'i, saklanan baytı ve süreyi karşılaştırır.

In [ ]:
%%writefile stage_e.py
import torch


class MemEffSiLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ...

    @staticmethod
    def backward(ctx, g):
        ...


def mem_eff_silu(x):
    return MemEffSiLU.apply(x)


x = torch.randn(4, requires_grad=True)
mem_eff_silu(x).sum().backward()
print(x.grad)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Kaynaklar

- [PyTorch Docs — Extending torch.autograd](https://docs.pytorch.org/docs/stable/notes/extending.html) — BSD-3-Clause
- [PyTorch Tutorials — Double Backward with Custom Functions / Fusing Convolution and Batch Norm](https://docs.pytorch.org/tutorials/intermediate/custom_function_double_backward_tutorial.html) — BSD-3-Clause
- [Bengio, Léonard, Courville — Estimating or Propagating Gradients Through Stochastic Neurons (STE)](https://arxiv.org/abs/1308.3432) — arXiv
- [Ganin & Lempitsky — Unsupervised Domain Adaptation by Backpropagation (gradient reversal)](https://arxiv.org/abs/1409.7495) — arXiv